### Gather unique values for city names in San Diego County

In [ ]:
fc = r'C:\DemoData\PYTS\San_Diego\SanDiego.gdb\GasStations'
# City exists, SANDAG_CITY will be created later
fldNames = ['CITY', "SANDAG_CITY"]
d = arcpy.da.Describe(fc)
domainName = 'SANDAGCities'

#### Create a field, domain, and add coded values

`LA JOLLA` is the original value

`LA_JOLLA` will be the coded

`La Jolla` will be the value

##### Create a new field for cities

In [ ]:
arcpy.management.AddField(
    in_table = d['catalogPath'],
    field_name = fldNames[1],
    field_type = "TEXT",
    field_alias = "City Name"
)



##### Create a new coded value domain

In [ ]:
arcpy.management.CreateDomain(
    in_workspace = d['path'],
    domain_name = domainName,
    domain_description = "Domain for cities in San Diego County",
    field_type = "TEXT",
    domain_type = "CODED"
)

#### Extract list of cities and add each city as a coded value in the domain

In [ ]:
src = arcpy.da.SearchCursor(fc, fldNames)
cities = set() #sets only hold unique values
for row in src:
    if row[0] != ' ':
        city = row[0]
        cities.add(city)
print(cities)
del src

In [ ]:
for city in cities:
    code = city.replace(' ', '_')
    print(code)
    value = city.title()
    print(value)
    arcpy.management.AddCodedValueToDomain(
        in_workspace = d['path'],
        domain_name = domainName,
        code = code,
        code_description = value)

#### Assign the newly created domain to the field

In [ ]:
arcpy.management.AssignDomainToField(
    in_table = d['catalogPath'],
    field_name = "SANDAG_CITY",
    domain_name = domainName,
    subtype_code = None
)

#### Edit the new, empty field using the domain codes

In [ ]:
# Query to exclude cities that have a space as the value
q = "CITY <> ' '"
uCur = arcpy.da.UpdateCursor(d['catalogPath'], ['CITY', 'SANDAG_CITY'], q)
for row in uCur:
    #originValue = row[0]
    #newValue = row[1]
    row[1] = row[0].replace(' ', '_') # transform the raw value into the coded value
    uCur.updateRow(row)
del uCur
print('Done')